# CorrosionAI grain classifier

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tian-haoyan/CorrosionAI/blob/main/colab/grain_classifier_demo.ipynb)

This notebook runs CorrosionAI inference on uploaded amphibole grain photomicrographs.

Recommended input images:

- one clearly visible amphibole grain per image;
- square, single-grain images, preferably 576 ? 576 pixels;
- RGB optical photomicrographs;
- limited background and limited overlap with other grains;
- no labels, arrows, scale bars, or text covering the grain.

Supported formats: png, jpg, jpeg, bmp, tif, tiff, webp.

## 1. Clone CorrosionAI and install dependencies

In [ ]:
!rm -rf CorrosionAI
!git clone https://github.com/Tian-haoyan/CorrosionAI.git
%cd CorrosionAI
!pip install -r requirements.txt

## 2. Download model checkpoint

The checkpoint is downloaded to `weights/best_acc.pth`. If automatic download fails, check that the Google Drive file is shared as "Anyone with the link can view", or upload the checkpoint manually into the `weights/` folder.

In [ ]:
import os
import gdown

os.makedirs("weights", exist_ok=True)

# Replace this file_id if the checkpoint is moved to a new Google Drive/Release/Zenodo link.
file_id = "1PH_-8IDaCeOTuoSGWGxcdn-xVC30lwof"
url = f"https://drive.google.com/uc?id={file_id}"
checkpoint_path = "weights/best_acc.pth"

gdown.download(url, checkpoint_path, quiet=False)
print("Checkpoint path:", checkpoint_path)
print("Checkpoint exists:", os.path.exists(checkpoint_path))

## 3. Upload images

This cell uploads images into one sample folder named `uploaded_sample`. If you want to analyse several samples separately, run the notebook once per sample or upload images through Google Drive using separate subfolders.

In [ ]:
from google.colab import files
import os

sample_name = "uploaded_sample"
input_root = "external_test_images"
sample_dir = os.path.join(input_root, sample_name)
os.makedirs(sample_dir, exist_ok=True)

uploaded = files.upload()
for filename, content in uploaded.items():
    with open(os.path.join(sample_dir, filename), "wb") as f:
        f.write(content)

print(f"Uploaded {len(uploaded)} files to {sample_dir}")
for fn in sorted(os.listdir(sample_dir))[:20]:
    print(fn)
if len(os.listdir(sample_dir)) > 20:
    print("...")

## 4. Run prediction and calculate CI*

In [ ]:
from pathlib import Path
import subprocess
import sys

input_dir = Path("external_test_images")
weights_path = Path("weights/best_acc.pth")
output_dir = Path("outputs/inference_results")

image_files = []
if input_dir.exists():
    image_files = [p for p in input_dir.rglob("*") if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"]]

print("Input folder:", input_dir)
print("Number of uploaded images:", len(image_files))
print("Checkpoint exists:", weights_path.exists())

if not weights_path.exists():
    raise FileNotFoundError("Model checkpoint was not found. Please rerun Step 2: Download model checkpoint.")
if len(image_files) == 0:
    raise FileNotFoundError("No input images were found. Please rerun Step 3: Upload images.")

cmd = [
    sys.executable,
    "inference/run_external_test.py",
    "--input", str(input_dir),
    "--weights", str(weights_path),
    "--output", str(output_dir),
    "--device", "auto",
]

print("Running inference...")
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Inference failed. Please copy the error message above and report it.")

predictions_csv = output_dir / "predictions.csv"
print("Predictions CSV exists:", predictions_csv.exists())
print("Predictions CSV path:", predictions_csv)
if not predictions_csv.exists():
    raise FileNotFoundError("Inference finished but predictions.csv was not created. Please report the log above.")


## 5. View prediction outputs

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path("outputs/inference_results")
predictions_csv = output_dir / "predictions.csv"

if not predictions_csv.exists():
    raise FileNotFoundError("predictions.csv was not found. Please run Step 4 first.")

pred = pd.read_csv(predictions_csv)
print("Prediction table. The final row contains the overall predicted CI* summary.")
display(pred)

print("
Overall predicted CI*:")
overall = pred.tail(1).iloc[0]
print(f"Total images: {overall['total_images']}")
print(f"Predicted CI*: {overall['predicted_CI_star']}")
print(
    "Predicted counts: "
    f"R={overall['n_R']}, A={overall['n_A']}, C={overall['n_C']}, "
    f"E={overall['n_E']}, S={overall['n_S']}"
)


## 6. Download all results

In [ ]:
!zip -r corrosionai_results.zip outputs/inference_results
from google.colab import files
files.download("corrosionai_results.zip")